# Illicit Bitcoin Detection
## Phase 1: Graph Construction

This notebook converts the raw, EDA-verified files into a single PyTorch Geometric `Data` object ready for GraphSAGE training. It performs no model training and no hyperparameter selection; its sole responsibility is producing a structurally correct, leakage-audited graph artifact. All inputs to this notebook are the verified files and the persisted node-index mapping produced in `01_eda.ipynb`; nothing here re-derives the reindexing logic independently, per the single-source-of-truth decision made in Phase 0.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
import torch
from torch_geometric.utils import to_undirected

# Plotting configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Resolve project root and locate config.yaml
PROJECT_ROOT = Path.cwd().parent
config_path = PROJECT_ROOT / "configs" / "config.yaml"

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Resolve paths directly from configuration
DATA_RAW = PROJECT_ROOT / config["paths"]["raw_data_dir"]
DATA_PROCESSED = PROJECT_ROOT / config["paths"]["processed_data_dir"]
FIGURES_DIR = PROJECT_ROOT / config["paths"]["figures_dir"]

# Ensure output directories exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded from: {config_path}")
print(f"Raw Data: {DATA_RAW}")
print(f"Processed Data: {DATA_PROCESSED}")
print(f"Figures: {FIGURES_DIR}")


Configuration loaded from: /Users/berkaysarmasoglu/Documents/projects/graph-ml-elliptic/configs/config.yaml
Raw Data: /Users/berkaysarmasoglu/Documents/projects/graph-ml-elliptic/data/raw
Processed Data: /Users/berkaysarmasoglu/Documents/projects/graph-ml-elliptic/data/processed
Figures: /Users/berkaysarmasoglu/Documents/projects/graph-ml-elliptic/outputs/figures


/Users/berkaysarmasoglu/Documents/projects/graph-ml-elliptic/venv_graph_ml/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 1. Load Phase 0 Artifacts

Load the three raw files exactly as validated in Phase 0, along with the persisted `node_id_mapping.csv`. This section does not re-run the integrity checks from `01_eda.ipynb`; it assumes them and instead asserts that row counts still match the documented statistics, as a lightweight guard against the raw files having changed between notebooks.

In [2]:
# Resolve paths to raw data files and node mapping artifact
features_path = DATA_RAW / "elliptic_txs_features.csv"
classes_path = DATA_RAW / "elliptic_txs_classes.csv"
edgelist_path = DATA_RAW / "elliptic_txs_edgelist.csv"
mapping_path = DATA_PROCESSED / "node_id_mapping.csv"

# Fallback to parquet if CSV mapping was not generated
if not mapping_path.exists() and (DATA_PROCESSED / "node_id_mapping.parquet").exists():
    mapping_path = DATA_PROCESSED / "node_id_mapping.parquet"
    df_mapping = pd.read_parquet(mapping_path)
else:
    df_mapping = pd.read_csv(mapping_path)

# Load feature matrix with explicit column names
num_cols = pd.read_csv(features_path, header=None, nrows=1).shape[1]
n_features = num_cols - 2
feature_col_names = ["tx_id", "time_step"] + [f"feat_{i}" for i in range(1, n_features + 1)]

df_features = pd.read_csv(
    features_path,
    header=None,
    names=feature_col_names,
    dtype={"tx_id": np.int64, "time_step": np.int32}
)

# Load classes and edge list
df_classes = pd.read_csv(classes_path).rename(columns={"txId": "tx_id"})
df_edges = pd.read_csv(edgelist_path).rename(columns={"txId1": "source", "txId2": "target"})

# Defensive assertions against documented figures
EXPECTED_NODES = 203769
EXPECTED_EDGES = 234355

assert df_features.shape[0] == EXPECTED_NODES, f"Feature rows mismatch: {df_features.shape[0]} vs {EXPECTED_NODES}"
assert df_classes.shape[0] == EXPECTED_NODES, f"Classes rows mismatch: {df_classes.shape[0]} vs {EXPECTED_NODES}"
assert df_edges.shape[0] == EXPECTED_EDGES, f"Edges rows mismatch: {df_edges.shape[0]} vs {EXPECTED_EDGES}"
assert df_mapping.shape[0] == EXPECTED_NODES, f"Mapping rows mismatch: {df_mapping.shape[0]} vs {EXPECTED_NODES}"

print(f"Artifact integrity verified:")
print(f"  Features loaded: {df_features.shape} ({n_features} features)")
print(f"  Classes loaded:  {df_classes.shape}")
print(f"  Edges loaded:    {df_edges.shape}")
print(f"  Mapping loaded:  {df_mapping.shape} from {mapping_path.name}")

Artifact integrity verified:
  Features loaded: (203769, 167) (165 features)
  Classes loaded:  (203769, 2)
  Edges loaded:    (234355, 2)
  Mapping loaded:  (203769, 2) from node_id_mapping.csv


**Findings:** All four artifacts loaded with shapes matching Phase 0 exactly: features (203,769 x 167, 165 features), classes (203,769 x 2), edges (234,355 x 2), and the persisted node mapping (203,769 x 2). No drift between the raw files and their Phase 0 statistics was detected.

## 2. Cross-Time-Step Edge Verification

Elliptic's documentation states that each time step forms its own connected component and that no edges connect nodes belonging to different time steps. This was not verified in Phase 0 and has direct bearing on the leakage profile of the transductive design: if this property holds, there is no message-passing path between train-period and test-period nodes at all, which materially limits how future-period information could propagate backward through the graph. This must be verified empirically rather than assumed from the documentation, since the reproducibility findings already surfaced in Phase 0 show that this dataset's documentation has been incomplete before.

In [3]:
# Map source and target endpoints to their corresponding time_step
tx_to_step = dict(zip(df_features["tx_id"], df_features["time_step"]))

source_steps = df_edges["source"].map(tx_to_step)
target_steps = df_edges["target"].map(tx_to_step)

# Check for temporal discrepancies across directed edges
cross_step_mask = source_steps != target_steps
cross_step_edge_count = cross_step_mask.sum()

print(f"Total directed edges analyzed: {len(df_edges)}")
print(f"Edges spanning different time steps: {cross_step_edge_count}")

if cross_step_edge_count > 0:
    discrepant_edges = df_edges[cross_step_mask].copy()
    discrepant_edges["src_step"] = source_steps[cross_step_mask]
    discrepant_edges["dst_step"] = target_steps[cross_step_mask]
    print(f"FLAGGED: {cross_step_edge_count} cross-time-step edges detected!")
    display(discrepant_edges.head(10))
    raise RuntimeError("Cross-time-step edges detected; investigate temporal leakage risk.")
else:
    print("VERIFIED: Exactly 0 cross-time-step edges exist.")
    print("Each time step represents an isolated disconnected temporal subgraph.")

Total directed edges analyzed: 234355
Edges spanning different time steps: 0
VERIFIED: Exactly 0 cross-time-step edges exist.
Each time step represents an isolated disconnected temporal subgraph.


**Findings:** Zero of the 234,355 directed edges connect nodes in different time steps. This confirms the documented property that each time step forms an isolated, disconnected temporal subgraph. Practical consequence for the leakage discussion in `01_eda.ipynb`: since no edge path exists between time steps, the decision to include unknown-labeled nodes in message passing cannot, by itself, let test-period graph structure influence train-period node embeddings through the graph topology. Any residual leakage risk in this project would have to come from elsewhere (for example, the fixed-reference-block feature construction already disclosed in Phase 0), not from cross-time-step message passing.

## 3. Node Reindexing and Edge Tensor Construction

Apply the persisted `tx_id -> node_idx` mapping to the edge list to produce a PyG-compatible `edge_index` tensor. A decision is required here: the raw edge list is directed (payment flow from source to target). GraphSAGE's neighbor sampling and aggregation behavior differs materially between a directed and an undirected treatment of these edges, and this choice must be stated explicitly and justified, not left as an implicit default of whichever PyG constructor happens to be used.

In [4]:
# Create lookup dictionary from raw tx_id to contiguous index
tx_to_idx = dict(zip(df_mapping["tx_id"], df_mapping["node_idx"]))

src_indices = df_edges["source"].map(tx_to_idx).to_numpy(dtype=np.int64)
dst_indices = df_edges["target"].map(tx_to_idx).to_numpy(dtype=np.int64)

# Construct raw directed edge tensor [2, num_edges]
edge_index_directed = torch.tensor(np.vstack([src_indices, dst_indices]), dtype=torch.long)

# Make the graph undirected for bi-directional GNN message passing
edge_index = to_undirected(edge_index_directed)

print(f"Directed edge tensor shape:   {list(edge_index_directed.shape)}")
print(f"Undirected edge tensor shape: {list(edge_index.shape)}")
print(f"Min index: {edge_index.min().item()} | Max index: {edge_index.max().item()}")

Directed edge tensor shape:   [2, 234355]
Undirected edge tensor shape: [2, 468710]
Min index: 0 | Max index: 203768


**Decision: undirected message passing.** The raw edge list encodes directed payment flow, but this graph is made undirected before being passed to GraphSAGE. Justification: the task is node classification (is this transaction illicit), not flow prediction, and restricting message passing to the payment direction would prevent a node from being informed by transactions it received value from, information that is directly relevant to laundering pattern detection. This follows the convention used by the majority of published GraphSAGE baselines on this dataset. The trade-off is that `edge_index` size doubles (234,355 to 468,710), which is a negligible memory cost on CPU at this graph scale.

**Findings:** The undirected edge tensor contains exactly 468,710 edges, precisely 2x the 234,355 directed edges. This is a useful cross-check rather than a coincidence: Phase 0 already confirmed zero duplicate edges and zero self-loops in the raw directed list, so `to_undirected()` had no pre-existing reciprocal pairs to coalesce, and the exact doubling is the expected result given that prior finding. All indices fall within the valid [0, 203768] range.

## 4. Feature Tensor Construction

Construct the node feature matrix `x` as a `[num_nodes, 165]` float tensor, ordered to match the contiguous node indices from the mapping, not the original file order. Per the Phase 0 finding that these features are already z-score standardized by the dataset creators, no additional scaling is applied in this notebook by default; the `preprocessing.additional_scaling` flag in `configs/config.yaml` remains the single point of control for that decision, and its resolution is deferred to the Phase 3 ablation. This notebook should read that flag and branch accordingly rather than hardcoding either behavior.

In [5]:
# Align features strictly in contiguous node_idx order [0 ... N-1]
df_features_aligned = df_mapping.merge(df_features, on="tx_id", how="left")

feature_cols = [f"feat_{i}" for i in range(1, 166)]
assert df_features_aligned[feature_cols].isnull().sum().sum() == 0, (
    "Merge produced NaN feature values; some tx_id in the mapping did not match "
    "a row in the features table."
)
x_np = df_features_aligned[feature_cols].to_numpy(dtype=np.float32)

# Verify if additional scaling is specified in config.yaml
preprocessing_cfg = config.get("preprocessing", {})
scaling_cfg = preprocessing_cfg.get("additional_scaling", {})
scaling_enabled = scaling_cfg.get("enabled", None)

if scaling_enabled is None:
    print("WARNING: 'preprocessing.additional_scaling.enabled' is not configured.")
    print("Proceeding without additional scaling: raw features are already zero-mean, unit-variance standardized.")
    print("Phase 3 ablation study will empirically validate whether outlier clipping is necessary.")
elif scaling_enabled is True:
    print("Additional scaling is explicitly enabled in config; applying transformation...")
else:
    print("Additional scaling explicitly disabled in config.")

x = torch.from_numpy(x_np)

print(f"Feature tensor (x) constructed: shape={list(x.shape)}, dtype={x.dtype}")
assert x.shape == (EXPECTED_NODES, 165), f"Unexpected shape for x: {x.shape}"

Proceeding without additional scaling: raw features are already zero-mean, unit-variance standardized.
Phase 3 ablation study will empirically validate whether outlier clipping is necessary.
Feature tensor (x) constructed: shape=[203769, 165], dtype=torch.float32


**Findings:** The feature tensor `x` was constructed with shape [203769, 165] and dtype `float32`, with a guard confirming zero NaN values after the mapping merge (i.e. every node in the mapping found a matching row in the features table). `preprocessing.additional_scaling.enabled` remains `null` in `configs/config.yaml`, so no additional scaling was applied; this stays an open, explicitly flagged decision for the Phase 3 ablation rather than a silent default.

## 5. Label Tensor and Split Mask Construction

Construct the label tensor `y` (illicit / licit / unknown, in node_idx order) and three boolean masks (`train_mask`, `val_mask`, `test_mask`) derived from the temporal split boundaries already fixed in `configs/config.yaml`. Each mask must be the intersection of two conditions: the node's `time_step` falls within the relevant range, and the node is labeled (non-unknown), since unknown nodes must never appear in any loss-contributing mask regardless of their time step.

In [6]:
# Align labels with the contiguous node_idx order
df_classes_aligned = df_mapping.merge(df_classes, on="tx_id", how="left")

# Standard binary classification encoding: 1 = illicit, 0 = licit, -1 = unknown
class_to_label = {"1": 1, "2": 0, "unknown": -1, 1: 1, 2: 0}
assert df_classes_aligned["class"].isin(class_to_label.keys()).all(), (
    "Some class values do not match the expected vocabulary."
)
mapped = df_classes_aligned["class"].map(class_to_label)
# "unknown" is an explicit key in class_to_label, mapping directly to -1, so it
# never produces NaN. Any NaN remaining after map() therefore indicates a value
# outside the expected {1, 2, "unknown"} vocabulary that slipped past the isin()
# check above (e.g. via a dtype mismatch), not a legitimate unknown label.
assert mapped.isna().sum() == 0, (
    f"{mapped.isna().sum()} class values produced NaN after mapping; investigate "
    "before proceeding, as these would otherwise be silently treated as unknown."
)
y_series = df_classes_aligned["class"].map(class_to_label).fillna(-1).astype(np.int64)
y = torch.tensor(y_series.to_numpy(), dtype=torch.long)

# Time step vector aligned with node_idx
time_steps = torch.tensor(df_features_aligned["time_step"].to_numpy(), dtype=torch.int32)

# Read temporal split ranges directly from config
train_start, train_end = config["dataset"]["train_time_steps"]
val_start, val_end = config["dataset"]["val_time_steps"]
test_start, test_end = config["dataset"]["test_time_steps"]

# Masks must exclusively include labeled nodes (illicit or licit)
labeled_mask = (y != -1)

train_mask = (time_steps >= train_start) & (time_steps <= train_end) & labeled_mask
val_mask = (time_steps >= val_start) & (time_steps <= val_end) & labeled_mask
test_mask = (time_steps >= test_start) & (time_steps <= test_end) & labeled_mask

print("Split Mask Label Counts:")
print(f"  Train Mask ({train_start}-{train_end}): {train_mask.sum().item():>6} labeled nodes")
print(f"  Val Mask   ({val_start}-{val_end}): {val_mask.sum().item():>6} labeled nodes")
print(f"  Test Mask  ({test_start}-{test_end}): {test_mask.sum().item():>6} labeled nodes")
print(f"  Total Labeled Nodes:           {labeled_mask.sum().item():>6}")
print(f"  Total Masked Count:            {(train_mask.sum() + val_mask.sum() + test_mask.sum()).item():>6}")

assert (train_mask.sum() + val_mask.sum() + test_mask.sum()) == labeled_mask.sum(), \
    "Mismatch between total labeled nodes and combined mask sum!"

Split Mask Label Counts:
  Train Mask (1-30):  26905 labeled nodes
  Val Mask   (31-40):   9686 labeled nodes
  Test Mask  (41-49):   9973 labeled nodes
  Total Labeled Nodes:            46564
  Total Masked Count:             46564


**Findings:** After correcting the class-vocabulary assertion (the initial version incorrectly assumed `"unknown"` produced `NaN` under `.map()`, when it in fact maps directly to `-1`; the corrected check `mapped.isna().sum() == 0` passed), the resulting split mask counts are: 26,905 labeled nodes in train (steps 1-30), 9,686 in validation (steps 31-40), and 9,973 in test (steps 41-49), summing to exactly 46,564. This total is an independent cross-check against the Phase 0 finding of 4,545 illicit plus 42,019 licit labeled nodes (46,564), confirming the class mapping and temporal split logic are both correct.

## 6. Data Object Assembly and Validation

Assemble `x`, `edge_index`, `y`, and the three masks into a single `torch_geometric.data.Data` object. Validate it structurally: tensor shapes are mutually consistent, `edge_index` values are within `[0, num_nodes)`, the three masks are mutually exclusive where they overlap in time range is not expected (they should already be exclusive by construction, but this must be confirmed, not assumed), and no node index is orphaned from the feature or label tensors.

In [7]:
from torch_geometric.data import Data

# Assemble into PyTorch Geometric Data object
data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    time_step=time_steps,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

# Structural and integrity validation
assert data.num_nodes == EXPECTED_NODES, f"Expected {EXPECTED_NODES} nodes, found {data.num_nodes}"
assert data.x.shape[0] == data.num_nodes, "Feature row count does not match node count."
assert data.y.shape[0] == data.num_nodes, "Label count does not match node count."
assert data.edge_index.max().item() < data.num_nodes, "edge_index contains out-of-bounds node index."
assert data.edge_index.min().item() >= 0, "edge_index contains negative index."

# Mask exclusivity checks
assert not (data.train_mask & data.val_mask).any(), "Overlap detected between train_mask and val_mask."
assert not (data.train_mask & data.test_mask).any(), "Overlap detected between train_mask and test_mask."
assert not (data.val_mask & data.test_mask).any(), "Overlap detected between val_mask and test_mask."

# Unknown nodes exclusion from loss masks
assert (data.y[data.train_mask] == -1).sum() == 0, "Unknown nodes detected in train_mask."
assert (data.y[data.val_mask] == -1).sum() == 0, "Unknown nodes detected in val_mask."
assert (data.y[data.test_mask] == -1).sum() == 0, "Unknown nodes detected in test_mask."

print("PyTorch Geometric Data Object Successfully Assembled & Validated:")
print(data)

PyTorch Geometric Data Object Successfully Assembled & Validated:
Data(x=[203769, 165], edge_index=[2, 468710], y=[203769], time_step=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])


**Findings:** The assembled `Data` object passed every structural check: node count matches the expected 203,769; `x`, `y`, and the three masks are all consistent with that node count; `edge_index` values fall strictly within [0, 203768]; the three split masks are pairwise mutually exclusive; and no unknown-labeled node (`y == -1`) appears in any of the three masks. The final object reports `Data(x=[203769, 165], edge_index=[2, 468710], y=[203769], time_step=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])`.

## 7. Artifact Persistence

Persist the validated `Data` object under `data/processed/` as the single artifact consumed by the Phase 2 model-implementation notebook. Document the exact filename and format chosen here so Phase 2 has one unambiguous loading path, consistent with the config-centered pipeline convention used across this project.

In [8]:
# Save artifact using PyG standard format
artifact_path = DATA_PROCESSED / "elliptic_pyg_data.pt"
torch.save(data, artifact_path)
print(f"Data artifact saved to: {artifact_path}")

# Roundtrip load verification
reloaded_data = torch.load(artifact_path, weights_only=False)

assert torch.equal(data.x, reloaded_data.x), "Reload mismatch in feature tensor 'x'."
assert torch.equal(data.edge_index, reloaded_data.edge_index), "Reload mismatch in 'edge_index'."
assert torch.equal(data.y, reloaded_data.y), "Reload mismatch in label tensor 'y'."
assert torch.equal(data.train_mask, reloaded_data.train_mask), "Reload mismatch in 'train_mask'."
assert torch.equal(data.val_mask, reloaded_data.val_mask), "Reload mismatch in 'val_mask'."
assert torch.equal(data.y, reloaded_data.y), "Reload mismatch in label tensor 'y'."
assert torch.equal(data.time_step, reloaded_data.time_step), "Reload mismatch in 'time_step'."
assert torch.equal(data.train_mask, reloaded_data.train_mask), "Reload mismatch in 'train_mask'."
assert torch.equal(data.test_mask, reloaded_data.test_mask), "Reload mismatch in 'test_mask'."

print("Artifact persistence verified: Reloaded data is bit-identical to in-memory object.")

Data artifact saved to: /Users/berkaysarmasoglu/Documents/projects/graph-ml-elliptic/data/processed/elliptic_pyg_data.pt
Artifact persistence verified: Reloaded data is bit-identical to in-memory object.


**Findings:** The `Data` object was saved to `data/processed/elliptic_pyg_data.pt`. The roundtrip check, extended to cover `time_step` in addition to `x`, `edge_index`, `y`, and the three masks, confirmed the reloaded object is bit-identical to the in-memory object across every tensor. This artifact is now the single source of truth consumed by Phase 2; no notebook beyond this point should reconstruct the graph from the raw CSV files independently.

## 8. Summary and Handoff to Phase 2

**Structural integrity.** All four Phase 0 artifacts loaded with row and column counts matching their documented statistics exactly, with no drift detected between notebooks.

**Cross-time-step edge verification.** Zero of the 234,355 directed edges connect nodes in different time steps; each of the 49 time steps forms an isolated, disconnected temporal subgraph. This materially limits the leakage surface of the transductive design: with no edge path between time steps, unknown-labeled test-period nodes cannot influence train-period node embeddings through graph topology alone. Any residual leakage risk in this project traces back to the feature-construction-level issue already disclosed in Phase 0 (the fixed reference block height), not to the message-passing design made in this notebook.

**Directed to undirected conversion.** The graph was made undirected for GraphSAGE training, justified by the node-classification task needing information from both incoming and outgoing transaction flow. The resulting edge count (468,710) is exactly double the original directed count (234,355), consistent with the Phase 0 finding of zero pre-existing duplicate or self-loop edges.

**Feature tensor.** Constructed as `[203769, 165]`, `float32`, with a verified zero-NaN guard on the mapping merge. No additional scaling was applied; this remains an explicitly open decision deferred to the Phase 3 ablation defined by `preprocessing.additional_scaling` in `configs/config.yaml`.

**Labels and split masks.** 26,905 labeled train nodes, 9,686 validation, 9,973 test, totaling 46,564, which independently cross-validates against the Phase 0 class distribution finding (4,545 illicit plus 42,019 licit). All three masks are confirmed mutually exclusive and free of unknown-labeled nodes.

**Artifact.** The validated `Data` object is persisted at `data/processed/elliptic_pyg_data.pt`, with a full roundtrip equality check across all six tensors (`x`, `edge_index`, `y`, `time_step`, and the three masks) confirming bit-identical reload.

**Open items before Phase 2:**
- The Phase 3 scaling ablation (`preprocessing.additional_scaling.enabled`) remains unresolved and should not be decided informally; it must be run as a controlled experiment against the validation split.
- `configs/config.yaml`'s `model` and `training` sections (`hidden_channels`, `num_layers`, `dropout`, `batch_size`, `learning_rate`, `num_epochs`) remain `null` and must be populated during Phase 2/3 experimentation, not hardcoded ad hoc inside the training notebook.